# 15장 · PDF 로딩 · PdfReader로 텍스트와 출처 쪽수 읽기

PdfReader는 PDF 파일을 열어 페이지별로 읽는 객체입니다. pages에서 한 쪽을 고른 다음 extract_text()를 호출하면 그 쪽의 텍스트를 얻습니다.

## PDF 한 쪽의 텍스트 꺼내기

`PdfReader(경로)`로 파일을 엽니다. `pages[0]`은 첫 쪽이고, 그 쪽의 `extract_text()`가 텍스트를 문자열로 돌려줍니다. 아래 준비 셀은 실습 폴더를 찾아 `rag_dir`에 저장합니다.

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 커널에서 `%pip install pypdf python-dotenv`를 한 번 실행하세요. 이번 장에는 모델 키가 필요하지 않습니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))
from portfolio import load_step, save_step, page_id


In [2]:
from pypdf import PdfReader

book = PdfReader(rag_dir / "loan-summer.pdf")
print(book.pages[0].extract_text().splitlines()[0])

실습용 문서  |  물결 장비실


출력한 `실습용 문서  |  물결 장비실`은 첫 쪽에서 읽은 첫 줄입니다. `pages[0]`만 `pages[1]`로 바꾸면 같은 파일의 둘째 쪽을 읽습니다.

## PDF에서 상담에 쓸 페이지 모으기

들소리 녹음기 사용자가 “연속 녹음하면 파일이 나뉠 때 녹음도 멈추나요?”라고 물었습니다. A형 두 판본과 B형 설명서의 모든 쪽을 읽어 파일명·쪽수·본문을 보관하세요. 14장에서 만든 양식에 새 녹음기 질문과 선택한 PDF 원문을 넣습니다.

In [ ]:
from rag_lab import MANUALS, read_pages
from langchain_core.prompts import ChatPromptTemplate
saved_prompt = load_step(root, "prompt")
pages_for_chat = [row for file in MANUALS for row in read_pages(file)]
selected = next(row for row in pages_for_chat if row["source"] == "recorder-a-1.1.pdf" and row["page"] == 2)
reader_prompt = ChatPromptTemplate.from_messages([("human", saved_prompt["template"])])
request = reader_prompt.invoke({"question": "파일이 나뉠 때 녹음이 멈추나요?", "context": selected["text"]})
print(selected["id"], "45분" in request.to_messages()[0].content)
save_step(root, "pages", pages_for_chat)

recorder-a-1.1.pdf#p2 True


`selected["text"]`를 두 번째 빈칸에 넣었으므로 `True`가 나오면 1.1판 2쪽의 ‘45분’이 모델에 보낼 메시지에 들어갔습니다. 아직 모델 호출은 하지 않았습니다. `my-course/rag/pages.json`에는 이 페이지를 포함한 원문 여섯 쪽이 남습니다.

### 실습 · 구판 2쪽을 다시 찾기

목록에서 A형 1.0판 2쪽을 고르세요. 화면에 보이는 2쪽과 파일명이 맞고 본문에 ‘30분’이 있어야 합니다.

In [ ]:
# pages_for_chat에서 파일명과 쪽수를 모두 만족하는 항목을 찾으세요.

<details><summary>정답 코드·출력·풀이</summary><pre><code>old = next(r for r in pages_for_chat if r[&quot;source&quot;] == &quot;recorder-a-1.0.pdf&quot; and r[&quot;page&quot;] == 2)
print(old[&quot;id&quot;], &quot;30분&quot; in old[&quot;text&quot;])</code></pre><p>제공 자료로 확인할 값</p><pre>recorder-a-1.0.pdf#p2 True</pre><p>쪽수가 같아도 판본이 다릅니다. 16장이 읽는 JSON에는 두 파일의 출처가 모두 남습니다.</p></details>

## 응용 실습 · PDF에서 쪽수와 원문 읽기

## PdfReader.pages와 extract_text로 한 쪽 읽기

`PdfReader(경로)`는 PDF를 읽을 객체를 만듭니다. `book.pages`에서 한 쪽을 고른 뒤 `extract_text()`를 불러야 그 쪽의 글을 얻습니다. `pages[1]`의 1은 0부터 세는 위치이므로 사람이 펼칠 곳은 두 번째 쪽입니다. 출력에서 전체 쪽수 바로 다음 줄을 찾아 PDF의 두 번째 쪽 맨 위와 대조하세요.

자료의 `sources/recorder-a-1.0.pdf`와 `sources/recorder-a-1.1.pdf`를 열고 두 번째 쪽의 **연속 녹음**을 읽으세요. 3월 15일과 4월 15일의 파일 분할 간격을 각각 메모합니다. 같은 A형인데 값이 달라집니다. 판본과 적용 기간도 근거의 일부입니다.

이번에는 `rag/loan-summer.pdf`를 열어 신규 신청과 수령 시간만 바꿀 때의 규칙이 몇 쪽인지 찾으세요. 첫 쪽만 읽으면 이미 승인된 학생에게도 신규 신청을 요구하는 실수를 하게 됩니다.

In [2]:
from pypdf import PdfReader
from rag_lab import SOURCES

book = PdfReader(SOURCES / "recorder-a-1.1.pdf")
print("전체 쪽수", len(book.pages))
# Python의 두 번째 칸은 1번, 학생이 보는 PDF 쪽수는 2쪽입니다.
print(book.pages[1].extract_text())

전체 쪽수 2
들소리 A형   /   휴대용 녹음기
파일 저장과 복사
1.1판    적용 기간   2027년 4월 1일부터
연속 녹음
한 번에 길게 녹음하면 45분마다 새 파일이 열립니다. 파일이
바뀌어도 녹음은 계속됩니다.
파일 옮기기
녹음을 끝낸 뒤 케이블로 컴퓨터에 연결하세요. 녹음기에 있는
RECORD 폴더를 열고 필요한 파일을 컴퓨터로 복사합니다.
저장 공간이 부족할 때
저장 공간이 가득 차면 녹음이 멈춥니다. 이미 저장한 파일은
남습니다. 파일을 컴퓨터에 복사한 뒤 녹음기에서 지워 공간을
확보하세요.
가상 제품 설명서   A형 1.1판
2 / 2쪽



“파일이”와 “바뀌어도” 사이에 줄바꿈이 있습니다. 종이 페이지의 행을 나눈 흔적입니다. 숫자를 분리하거나 문서 전체의 줄을 무조건 지우면 제목과 본문 경계도 잃을 수 있습니다. 원문을 보며 이 페이지에서 **뜻이 이어지는 두 줄**과 **새 소제목**을 각각 골라 보세요.

`read_pages`는 이 교재의 `rag_lab.py`에 정의한 함수입니다. 한 PDF의 각 쪽에 번호를 1부터 붙이고, 파일 이름에서 읽은 기종·판본과 함께 딕셔너리 목록으로 돌려줍니다. 그래서 `pages[1]["page"]`는 2이고 `pages[1]["text"]`에는 그 쪽의 글이 들어갑니다. 이 둘을 함께 보관해야 검색한 글에서 원본 쪽으로 돌아갈 수 있습니다.

In [3]:
from rag_lab import read_pages

pages = read_pages(SOURCES / "recorder-a-1.1.pdf")
second = pages[1]
print(second["id"], second["page"], second["edition"])
print("45분" in second["text"])

recorder-a-1.1.pdf#p2 2 1.1
True


## read_pages의 반환값에서 ID와 쪽수 읽기

`read_pages(경로)`는 쪽마다 딕셔너리 한 개를 만들어 리스트로 돌려주는 수업용 함수입니다. `pages[1]`은 리스트의 두 번째 항목이고 `pages[1]["page"]`는 학생에게 표시할 쪽수 2입니다. 같은 항목의 `id`에는 파일명과 쪽수가 함께 들어 있습니다. 서로 다른 PDF의 2쪽을 구별할 수 있는 이유입니다.

## 여러 PDF에서 같은 문장이 나온 쪽의 ID 찾기

“녹음은 계속됩니다”라는 말을 추출한 전 페이지에서 찾아 봅니다. 하나만 찾으면 이미 확인이 끝난 것 같지만 이 문구는 기종·판본 여러 곳에 반복됩니다. 아래 출력은 ‘페이지가 같은 문장을 담았는가’만 증명합니다. 그중 이용자의 조건에 맞는 페이지를 고르는 판단은 별도로 필요합니다.

In [4]:
from rag_lab import MANUALS
for file in MANUALS:
    for page in read_pages(file):
        if "녹음은 계속됩니다" in page["text"]:
            print(page["id"])

recorder-a-1.0.pdf#p2
recorder-a-1.1.pdf#p2
recorder-b-1.0.pdf#p2


3월 A형의 연속 녹음을 묻는다면 세 쪽 중 A형 1.0판 2쪽을 엽니다. 다음에는 “녹음이 멈춥니다”를 찾아 보고 그 말이 파일 분할을 뜻하는지 저장 공간 부족을 뜻하는지 앞 문장을 읽으세요. 같은 주제 안에서도 조건을 함께 찾아야 결론을 뒤집지 않습니다.

앞표지와 인쇄 쪽수가 따로 있는 책에서는 화면의 PDF 위치와 본문 표기 쪽수를 별도 칸에 둡니다. 이 수업용 설명서는 표지가 없어 첫 PDF 페이지와 인쇄 1쪽이 같습니다. 나중에 팀 문서를 받으면 첫 쪽과 끝 쪽을 함께 열어 이 가정을 다시 확인하세요.

### 실습 · PDF 둘째 쪽을 다시 찾을 값 출력하기

예제에서 읽은 pages의 두 번째 항목에서 파일명, 쪽수, ID를 출력하세요. 파일명이 같아도 다른 쪽을 구별할 수 있는 값이 어느 것인지 확인하세요.

In [ ]:
# 위에서 만든 pages를 사용하세요.

<details><summary>정답 코드와 실행 결과</summary><pre><code>entry = pages[1]
print(entry[&quot;source&quot;], entry[&quot;page&quot;])
print(entry[&quot;id&quot;])</code></pre><pre>recorder-a-1.1.pdf 2
recorder-a-1.1.pdf#p2</pre><p>리스트 위치는 1이지만 사람이 보는 쪽수는 2입니다. ID에 두 값이 함께 들어 있어 같은 파일에서도 원하는 쪽으로 돌아갈 수 있습니다.</p></details>

## 실습 · 예외 조항이 있는 쪽 채집하기

loan-summer.pdf에서 ‘승인된 신청의 수령 시간만 변경’과 ‘장비 종류 변경’을 비교하세요. 파일·쪽·판본과 두 문장을 붙인 근거 카드를 작성합니다. 한 쪽에 여러 단락이 들어 있어도 해당 문장을 좁혀 표시하세요.

In [5]:
guide = PdfReader(rag_dir / "loan-summer.pdf")
screen_page = 2
# 화면 쪽수를 pages의 인덱스로 바꾸고 extract_text()로 해당 쪽을 읽으세요.
# 시간만 바꿀 때와 장비 종류까지 바꿀 때의 문장을 각각 찾으세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>guide = PdfReader(rag_dir / &quot;loan-summer.pdf&quot;)
print(&quot;loan-summer.pdf 2쪽&quot;)
print(guide.pages[1].extract_text())</code></pre><p>둘 다 1.1판 2쪽입니다. 시간만 바꾸면 전날 오후 4시까지 알리고, 장비 종류까지 바꾸면 신규 신청 절차입니다. 같은 쪽을 찾은 뒤에도 대상을 읽어야 합니다.</p><p>집필 시 실행 결과</p><pre>loan-summer.pdf 2쪽
실습용 문서  |  물결 장비실
변경과 예외
장비 대여 안내  1.1판  |  2027년 4월 1일부터
이미 승인받은 장비의 수령 시간만 바꾸는 경우에는
위 신규 신청 마감을 다시 적용하지 않습니다.
수령 전날 오후 4시까지 담당자에게 새 시간을 알려 주세요.
장비 종류까지 바꾸려면 신규 신청 절차를 따릅니다.
교육용 가상 기관    물결 장비실     1.1판 / 2쪽</pre></details>

## 새로운 문서를 받았을 때

첫 쪽의 제목과 마지막 쪽의 예외를 직접 복사해 눈으로 비교하세요. 빈 문자열이 나오면 화면에 보이는 글자가 텍스트로 들어 있는지 확인합니다. 이미지로만 된 스캔본은 별도의 문자 인식이 필요할 수 있습니다. 추출 결과가 비었는데 빈 페이지를 색인에 넣으면 뒤에서 검색 설정만 고치게 됩니다.

수업 함수 `read_pages`는 녹음기 예제의 파일명에서 기종과 판본을 읽습니다. 이름 형식이 다른 PDF를 처음 점검할 때는 `PdfReader(경로).pages[0].extract_text()`로 글부터 확인하고, 출처 정보는 그 문서에 맞게 정하세요.

## 독립 실습 · 상담 메모의 출처 복구

담당자의 메모에는 “A형은 30분, 파일은 계속 저장된다”라고만 적혀 있습니다. 2027년 4월 문의자가 이 답을 받아도 될지 검수하세요. 자료 폴더에서 원본 두 개를 열고 **구판에서도 맞는 부분**, **판본이 바뀌어 틀린 부분**, **고쳐 쓸 문장**을 쪽수와 함께 적으세요. 동료에게 파일명과 쪽수만 건네 같은 문장을 찾게 합니다.

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>3월에 적용된 A형 1.0판 2쪽은 30분, 4월부터 적용된 1.1판 2쪽은 45분입니다. 파일이 바뀌어도 녹음이 계속된다는 사실은 두 판에서 같습니다. 4월 사용자에게는 “45분마다 새 파일이 열리고 녹음은 이어집니다”라고 답하고 A형 1.1판 2쪽을 제시합니다.</p></details>

## 다음 수업에 가져갈 것

16장을 열기 전에 실습 묶음의 `my-course/rag/pages.json`이 있는지 확인하세요. 각 쪽의 원문과 파일명·쪽수·판본이 함께 들어 있습니다.

A형 두 판본의 파일명·쪽수·분할 시간과, 대여 안내의 예외 문장 카드를 보관하세요. 다음 장에서는 쪽수를 잃지 않으면서 긴 본문을 검색하기 좋은 크기로 나눕니다.

편람의 13쪽과 35쪽을 추출해 보세요. 13쪽에서 제안한 시각이 결정되었는지, 35쪽에 적힌 시각이 서비스 기한인지 원문에서 근거를 찾습니다. [36쪽 운영 편람 연속 실습](pre-mid/documents/transfer/handbook-lab.md)